# Week 1 &mdash; Geometric Priors and Symmetry

## Implementation: Symmetry Groups and Equivariance in Code

This notebook accompanies the Week 1 theory material. We construct symmetry-group actions numerically, *empirically verify* invariance and equivariance, and implement a minimal equivariant layer. The goal is to build operational intuition for the abstract definitions.

### Setup


In [ ]:
# If running on a fresh environment, uncomment to install dependencies.
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as Rot

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(42)


## 1. The Cyclic Group $C_4$ Acting on Images

The simplest discrete symmetry of a square image is the cyclic group $C_4 = \{e, r, r^2, r^3\}$ of $90^\circ$ rotations. We implement the group action and verify the group axioms.


In [ ]:
def group_action_c4(img, k):
    """Action of the group element r^k (k in {0,1,2,3}) on an image."""
    return np.rot90(img, k % 4)

# A non-symmetric test image so that rotations are visibly distinct.
img = np.arange(16).reshape(4, 4).astype(float)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for k, ax in enumerate(axes):
    ax.imshow(group_action_c4(img, k), cmap="viridis")
    ax.set_title(f"$r^{k}$ applied")
    ax.axis("off")
plt.suptitle("The cyclic group $C_4$ acting on a 4x4 signal")
plt.tight_layout()
plt.show()

# Closure: r^2 . r^3 == r^(5 mod 4) == r^1
lhs = group_action_c4(group_action_c4(img, 3), 2)
rhs = group_action_c4(img, 1)
print("Closure r^2 r^3 = r^1 ?", np.allclose(lhs, rhs))


## 2. Empirically Testing Invariance vs. Equivariance

We compare two feature maps under the $C_4$ action:

- **Mean intensity** $\;\phi_{\text{inv}}(x) = \tfrac{1}{|\Omega|}\sum_u x(u)$ &mdash; expected to be *invariant*.
- **The signal itself under a learned filter** &mdash; expected to be *equivariant* only if the filter is rotationally symmetric.


In [ ]:
def mean_intensity(img):
    return img.mean()

# Invariance test for mean intensity.
vals = [mean_intensity(group_action_c4(img, k)) for k in range(4)]
print("Mean intensity over orbit:", np.round(vals, 4))
print("Invariant?", np.allclose(vals, vals[0]))


In [ ]:
from scipy.ndimage import convolve

def apply_filter(img, w):
    return convolve(img, w, mode="constant", cval=0.0)

# An ISOTROPIC (rotationally symmetric) filter -> equivariant convolution.
iso = np.array([[0, 1, 0],
                [1, 4, 1],
                [0, 1, 0]], dtype=float)
iso /= iso.sum()

# An ANISOTROPIC filter -> NOT equivariant under rotation.
aniso = np.array([[0, 0, 0],
                  [1, 0, -1],
                  [0, 0, 0]], dtype=float)

def equivariance_error(img, w):
    """max_k || f(r^k x) - r^k f(x) ||"""
    base = apply_filter(img, w)
    errs = []
    for k in range(4):
        lhs = apply_filter(group_action_c4(img, k), w)
        rhs = group_action_c4(base, k)
        errs.append(np.abs(lhs - rhs).max())
    return max(errs)

print("Equivariance error (isotropic filter):  ", equivariance_error(img, iso))
print("Equivariance error (anisotropic filter):", equivariance_error(img, aniso))


The isotropic filter yields a (near-)zero equivariance error: convolution with a rotationally symmetric kernel commutes with $C_4$. The anisotropic filter, sensitive to orientation, breaks equivariance &mdash; a direct demonstration that *the symmetry of the layer must match the symmetry we wish to respect*.


## 3. Building a Minimal Permutation-Equivariant Layer

Looking ahead to graphs (Weeks 2&ndash;3), the relevant symmetry is the permutation group $S_n$ acting on node features $X \in \mathbb{R}^{n \times d}$. A layer of the form
$$
f(X) = \sigma\big(X W_1 + \mathbf{1}\mathbf{1}^\top X\, W_2\big)
$$
(a learnable combination of each node's own features and the *aggregate* of all features) is permutation-equivariant. We verify this.


In [ ]:
def deep_sets_layer(X, W1, W2):
    """Permutation-equivariant layer: combines self and global-sum features."""
    agg = X.sum(axis=0, keepdims=True)          # global pooling (1, d)
    return np.tanh(X @ W1 + agg @ W2)           # broadcast over nodes

n, d, h = 5, 3, 4
X = rng.normal(size=(n, d))
W1 = rng.normal(size=(d, h))
W2 = rng.normal(size=(d, h))

# A random permutation matrix P.
perm = rng.permutation(n)
P = np.eye(n)[perm]

lhs = deep_sets_layer(P @ X, W1, W2)   # f(PX)
rhs = P @ deep_sets_layer(X, W1, W2)   # P f(X)
print("Permutation-equivariance error:", np.abs(lhs - rhs).max())


The error is at the level of floating-point precision, confirming that the Deep Sets layer is exactly permutation-equivariant. This construction is the conceptual seed of the message-passing networks we study in Week 3.


## 4. Continuous Symmetry: Sampling and Acting with $SO(3)$

Finally, we work with the continuous rotation group on a 3D point cloud, anticipating Week 4. We sample uniformly from $SO(3)$, apply rotations, and confirm that pairwise distances &mdash; a natural invariant feature &mdash; are preserved.


In [ ]:
# A point cloud sampled on the unit sphere.
N = 200
cloud = rng.normal(size=(N, 3))
cloud /= np.linalg.norm(cloud, axis=1, keepdims=True)

def pairwise_dists(P):
    diff = P[:, None, :] - P[None, :, :]
    return np.sqrt((diff**2).sum(-1))

D0 = pairwise_dists(cloud)

# Apply 10 random rotations and check invariance of the distance matrix.
max_err = 0.0
for s in range(10):
    R = Rot.random(random_state=s).as_matrix()
    D = pairwise_dists(cloud @ R.T)
    max_err = max(max_err, np.abs(D - D0).max())

print(f"Max change in pairwise distances over 10 random SO(3) rotations: {max_err:.2e}")


In [ ]:
# Visualise one rotation of the point cloud.
R = Rot.from_euler("xyz", [30, 45, 60], degrees=True).as_matrix()
rotated = cloud @ R.T

fig = plt.figure(figsize=(9, 4))
for i, (P, title) in enumerate([(cloud, "Original"), (rotated, "Rotated by SO(3) element")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.scatter(P[:, 0], P[:, 1], P[:, 2], s=8, alpha=0.6)
    ax.set_title(title)
    ax.set_box_aspect((1, 1, 1))
plt.tight_layout()
plt.show()


## 5. Summary and Exercises

We have, in code:

- Implemented the action of a finite group ($C_4$) and a continuous group ($SO(3)$) on signals.
- Empirically separated **invariant** features (mean intensity, pairwise distances) from **equivariant** maps (isotropic convolution, Deep Sets layer).
- Demonstrated that equivariance holds *only* when the layer's symmetry matches the data's symmetry.

### Exercises

1. Modify `deep_sets_layer` to also include a *max*-pooling aggregation. Is the result still permutation-equivariant? Verify.
2. Construct a $5\times 5$ anisotropic filter and measure how its equivariance error grows with anisotropy.
3. Implement the *invariant* read-out $\rho(x) = \tfrac{1}{n}\sum_i f(X)_i$ on top of the Deep Sets layer and confirm it is permutation-*invariant*.
